### Question 9

In [113]:
# Importing the necessary libraries
from statsmodels.stats import outliers_influence
from ISLP import load_data
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV, Ridge, Lasso
from sklearn.decomposition import PCA
from sklearn.cross_decomposition import PLSRegression
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
import statsmodels.api as sm
import numpy as np
from sklearn.model_selection import GridSearchCV

In [114]:

# Part (a): Split the data set into a training set and a test set

# Load the dataset
college=load_data("College")

# Define the features (X) and target variable (y)
from ISLP.models import (ModelSpec as MS, summarize, poly)

X = MS(college.drop(columns=['Apps', 'Private'])).fit_transform(college) # All columns except the target 'Apps' and 'Private'
y = college['Apps']  # Target variable is 'Apps'

# Split the data into training and testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)


In [115]:

# Part (b): Fit a linear model using least squares on the training set, and report the test error obtained

lm = LinearRegression()
lm.fit(X_train, y_train)
linear_model=sm.OLS(y_train,X_train)
linear_model.fit().summary()


<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                   Apps   R-squared:                       0.935
Model:                            OLS   Adj. R-squared:                  0.933
Method:                 Least Squares   F-statistic:                     543.9
Date:                Sun, 22 Sep 2024   Prob (F-statistic):               0.00
Time:                        22:08:19   Log-Likelihood:                -5190.4
No. Observations:                 621   AIC:                         1.041e+04
Df Residuals:                     604   BIC:                         1.049e+04
Df Model:                          16                                         
Covariance Type:            nonrobust                                         
===============================================================================
                  coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------
intercept    -488.2122    439.450     -1.111      0.267   -1351.248     374.824
Accept          1.6321      0.044     37.152      0.000       1.546       1.718
Enroll         -1.0706      0.204     -5.259      0.000      -1.470      -0.671
Top10perc      58.4482      6.322      9.245      0.000      46.032      70.864
Top25perc     -18.9681      5.055     -3.752      0.000     -28.896      -9.040
F.Undergrad     0.0896      0.036      2.504      0.013       0.019       0.160
P.Undergrad     0.0651      0.034      1.916      0.056      -0.002       0.132
Outstate       -0.1030      0.020     -5.055      0.000      -0.143      -0.063
Room.Board      0.1512      0.054      2.788      0.005       0.045       0.258
Books           0.0076      0.256      0.030      0.976      -0.494       0.510
Personal       -0.0126      0.071     -0.178      0.859      -0.151       0.126
PhD            -8.5598      5.265     -1.626      0.104     -18.899       1.779
Terminal       -1.0593      5.692     -0.186      0.852     -12.238      10.119
S.F.Ratio      14.7210     14.025      1.050      0.294     -12.823      42.265
perc.alumni    -1.9591      4.711     -0.416      0.678     -11.211       7.293
Expend          0.0553      0.014      4.050      0.000       0.028       0.082
Grad.Rate       7.9389      3.341      2.377      0.018       1.378      14.500
==============================================================================
Omnibus:                      373.032   Durbin-Watson:                   2.133
Prob(Omnibus):                  0.000   Jarque-Bera (JB):             7022.046
Skew:                           2.285   Prob(JB):                         0.00
Kurtosis:                      18.827   Cond. No.                     1.75e+05
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 1.75e+05. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

In [116]:

# Predicting on the test set and calculating test error (MSE)
y_pred_linear = lm.predict(X_test)
test_error_linear = mean_squared_error(y_test, y_pred_linear)
print(f'Linear Model Test Error (MSE): {test_error_linear}')


Linear Model Test Error (MSE): 1182113.6667499312


In [118]:
# Part (c): Fit a Ridge regression model on the training set, with λ chosen by cross-validation

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split the data into training and testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=0)

# Perform cross-validation to find the best lambda (alpha) for Ridge regression
alphas = np.logspace(-4, 4, 100)  # Lambda values to test
ridge_cv = Ridge()

# Grid search for the best lambda (alpha) using 5-fold cross-validation
grid_search = GridSearchCV(ridge_cv, param_grid={'alpha': alphas}, scoring='neg_mean_squared_error', cv=5)
grid_search.fit(X_train, y_train)

# Optimal lambda (alpha) from cross-validation
best_alpha = grid_search.best_params_['alpha']
print(f"Best alpha from cross-validation: {best_alpha}")

ridge_best = Ridge(alpha=best_alpha)
ridge_best.fit(X_train, y_train)

ridge_coef = ridge_best.coef_  # Coefficients of the Ridge model

# Predicting on the test set and calculating test error (MSE)
y_pred_ridge = ridge_best.predict(X_test)
test_error_ridge = mean_squared_error(y_test, y_pred_ridge)
print(f'Ridge Non zero coefficients: {ridge_coef}')
print(f'Ridge Model Test Error (MSE): {test_error_ridge}')



Best alpha from cross-validation: 0.0001
Ridge Non zero coefficients: [ 0.00000000e+00  3.99800093e+03 -9.94108692e+02  1.03038078e+03
 -3.75415896e+02  4.34340119e+02  9.91152665e+01 -4.13927090e+02
  1.65734625e+02  1.25845920e+00 -8.49963280e+00 -1.39675526e+02
 -1.55859334e+01  5.82333554e+01 -2.42620963e+01  2.88406120e+02
  1.36284907e+02]
Ridge Model Test Error (MSE): 1182112.48094328


In [119]:

# Part (d): Fit a Lasso model on the training set, with λ chosen by cross-validation
# Use GridSearchCV to find the best alpha (regularization strength)
grid_search1 = GridSearchCV(Lasso(), param_grid={'alpha': alphas}, cv=5, scoring='neg_mean_squared_error', verbose=1)
grid_search1.fit(X_train, y_train)

# Best alpha found by GridSearchCV
best_alpha1 = grid_search1.best_params_['alpha']
print(f"Best alpha from cross-validation: {best_alpha1}")


lasso_best = Lasso(alpha=best_alpha1)
lasso_best.fit(X_train, y_train)

lasso_coef = lasso_best.coef_  # Coefficients of the Lasso model

# Predicting on the test set and calculating test error (MSE)
y_pred_lasso = lasso_best.predict(X_test)
test_error_lasso = mean_squared_error(y_test, y_pred_lasso)
print(f'Lasso Non zero coefficients: {lasso_coef}')
print(f'Lasso Model Test Error (MSE): {test_error_lasso}')


Fitting 5 folds for each of 100 candidates, totalling 500 fits
Best alpha from cross-validation: 0.0001
Lasso Non zero coefficients: [ 0.00000000e+00  3.99800497e+03 -9.94112644e+02  1.03038179e+03
 -3.75416520e+02  4.34340281e+02  9.91152472e+01 -4.13927420e+02
  1.65734214e+02  1.25833901e+00 -8.49939123e+00 -1.39675640e+02
 -1.55856577e+01  5.82330934e+01 -2.42615996e+01  2.88405635e+02
  1.36284492e+02]
Lasso Model Test Error (MSE): 1182113.1539311882


In [120]:
# Part (e): Fit a PCR model on the training set, with M chosen by cross-validation

# Standardize the data for PCA
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Applying PCA with cross-validation to choose the number of components (M)
pca = PCA()
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

# Trying different number of components and using cross-validation to select the best M
mse_list = []
for M in range(1, X_train_pca.shape[1] + 1):
    pca_model = LinearRegression()
    mse = -cross_val_score(pca_model, X_train_pca[:, :M], y_train, cv=5, scoring='neg_mean_squared_error').mean()
    mse_list.append(mse)

# Selecting the best number of components (M)
best_M = mse_list.index(min(mse_list)) + 1

# Fit the linear model using the best number of components (PCR)
pca_model = LinearRegression()
pca_model.fit(X_train_pca[:, :best_M], y_train)

# Predicting on the test set
y_pred_pca = pca_model.predict(X_test_pca[:, :best_M])
test_error_pca = mean_squared_error(y_test, y_pred_pca)
print(f'PCR Model Test Error (MSE): {test_error_pca}, Best M: {best_M}')


PCR Model Test Error (MSE): 1182113.666749989, Best M: 17


In [121]:

# Part (f): Fit a PLS model on the training set, with M chosen by cross-validation

# Cross-validation to select the best number of components for PLS
pls_errors = []
for M in range(1, X_train.shape[1] + 1):
    pls_model = PLSRegression(n_components=M)
    mse = -cross_val_score(pls_model, X_train_scaled, y_train, cv=5, scoring='neg_mean_squared_error').mean()
    pls_errors.append(mse)

# Select the best number of components (M) for PLS
best_M_pls = pls_errors.index(min(pls_errors)) + 1

# Fit the PLS model with the best number of components
pls_model = PLSRegression(n_components=best_M_pls)
pls_model.fit(X_train_scaled, y_train)

# Predicting on the test set
y_pred_pls = pls_model.predict(X_test_scaled)
test_error_pls = mean_squared_error(y_test, y_pred_pls)
print(f'PLS Model Test Error (MSE): {test_error_pls}, Best M: {best_M_pls}')

PLS Model Test Error (MSE): 1195665.5907414078, Best M: 9
